In [32]:
# Comment out the following line to run the notebook in CPU mode
import plotly.express as px
import pandas as pd

# Suppress warnings
import warnings
from IPython.display import display
import json
import os
from tqdm import tqdm
from pathlib import Path
import numpy as np

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*column_view.*")

output_dir = Path() / "ancova_ieee"
output_dir.mkdir(parents=True, exist_ok=True)
print("Output directory:", output_dir.absolute())

Output directory: /mnt/lustre/ychatel/living-park/VIP-python-client/example/freesurfer-fuzzy/notebooks/ancova_ieee


In [33]:
def get_cohort_stat():
    df_cohort = pd.read_csv("../cohort_stat.csv")
    print(f"Load cohort stats: {os.path.abspath('../cohort_stat.csv')}")
    columns = [
        "PATNO",
        "first_visit",
        "second_visit",
        "dx_group",
        "SEX",
        "AGE_AT_VISIT",
        "durationT2_T1_y",
    ]
    df_cohort["first_visit"] = (
        "sub-"
        + df_cohort["PATNO"].astype(str)
        + "_ses-"
        + df_cohort["EVENT_ID"].astype(str)
    )
    df_cohort["second_visit"] = (
        "sub-"
        + df_cohort["PATNO"].astype(str)
        + "_ses-"
        + df_cohort["NEXT_VISIT"].astype(str)
    )
    # Remove PD-MCI subjects
    df_cohort = df_cohort[df_cohort["dx_group"] != "PD-MCI"]
    print(f"Number of PD-non-MCI subjects: {df_cohort.shape[0]}")
    return df_cohort[columns]


df_cohort = get_cohort_stat()


Load cohort stats: /mnt/lustre/ychatel/living-park/VIP-python-client/example/freesurfer-fuzzy/cohort_stat.csv
Number of PD-non-MCI subjects: 270


In [34]:
df_cohort

,PATNO,first_visit,second_visit,dx_group,SEX,AGE_AT_VISIT,durationT2_T1_y
27,100005,sub-100005_ses-BL,sub-100005_ses-V04,PD-non-MCI,1,52.8,1.016426
28,100006,sub-100006_ses-BL,sub-100006_ses-V04,PD-non-MCI,0,55.7,0.996058
29,100007,sub-100007_ses-BL,sub-100007_ses-V04,PD-non-MCI,1,67.2,1.110710
30,100012,sub-100012_ses-BL,sub-100012_ses-V04,PD-non-MCI,0,66.1,0.930355
31,100017,sub-100017_ses-BL,sub-100017_ses-V04,PD-non-MCI,0,58.8,0.976894
...,...,...,...,...,...,...,...
292,70345,sub-70345_ses-BL,sub-70345_ses-V06,HC,1,50.4,2.382611
293,72138,sub-72138_ses-BL,sub-72138_ses-V06,HC,0,55.5,1.791064
294,73940,sub-73940_ses-BL,sub-73940_ses-V06,HC,0,59.8,2.379873
295,75149,sub-75149_ses-V06,sub-75149_ses-V10,HC,0,74.6,1.916667


# ANCOVA

## Cortical

In [ ]:
def read_table(filename, hemi, measure):
    df = pd.read_csv(f"table_ieee/{hemi}.aparc.{measure}.tsv", sep="\t")
    df["hemi"] = hemi
    df.columns = [c.replace(f"{hemi}.", "") for c in df.columns]
    df.columns = [c.replace(f"{hemi}_", "") for c in df.columns]
    df.columns = [c.replace(f"_{measure}", "") for c in df.columns]
    df.rename(columns={f"aparc.{measure}": "PATNO_id"}, inplace=True)
    return df


def read_measure(measure):
    lh = read_table(f"table_ieee/lh.aparc.{measure}.tsv", "lh", measure)
    rh = read_table(f"table_ieee/rh.aparc.{measure}.tsv", "rh", measure)
    return pd.concat([lh, rh], axis=0)


def get_metric_visit(metric, cohort_df, visit):
    # Validate visit parameter
    if visit not in [1, 2]:
        raise ValueError("Visit must be 1 or 2")

    df = read_measure(metric)
    id_vars = ["PATNO_id", "hemi"]
    df = df.melt(id_vars=id_vars, var_name="region", value_name=metric)

    visit_col = "first_visit" if visit == 1 else "second_visit"

    clinical_columns = [
        visit_col,
        "AGE_AT_VISIT",
        "SEX",
        "durationT2_T1_y",
        "dx_group",
    ]

    merged_df = pd.merge(
        df,
        cohort_df[clinical_columns],
        left_on="PATNO_id",
        right_on=visit_col,
        how="inner",
    )

    # Clean up data types
    numeric_cols = [metric, "AGE_AT_VISIT", "durationT2_T1_y"]
    for col in numeric_cols:
        if col in merged_df.columns:
            merged_df[col] = pd.to_numeric(merged_df[col], errors="coerce")

    return merged_df


def get_longitudinal_metric(metric, cohort_df):
    baseline_df = get_metric_visit(metric, cohort_df=cohort_df, visit=1)
    next_df = get_metric_visit(metric, cohort_df=cohort_df, visit=2)

    if baseline_df.empty:
        raise ValueError("No baseline data available")
    if next_df.empty:
        raise ValueError("No longitudinal data available")

    baseline_df["PATNO"] = baseline_df["PATNO_id"].str.split("_").str[0]
    next_df["PATNO"] = next_df["PATNO_id"].str.split("_").str[0]

    # Compute change
    columns_to_merge = ["PATNO", "region", "hemi"]

    change_df = pd.merge(
        baseline_df,
        next_df,
        on=columns_to_merge,
        suffixes=("_baseline", "_next"),
    )

    if change_df.empty:
        raise ValueError("No matching records found between baseline and next visit")

    change_df[f"{metric}_change"] = (
        change_df[f"{metric}_next"] - change_df[f"{metric}_baseline"]
    ) / change_df[f"{metric}_baseline"]

    change_df.drop(columns=change_df.filter(regex="_baseline$").columns, inplace=True)
    change_df.rename(columns=lambda x: x.replace("_next", ""), inplace=True)

    return change_df

In [36]:
import pingouin as pg


def compute_ancova(measure, cohort_df, force):
    df = get_longitudinal_metric(measure, cohort_df)

    ancova_df = pd.DataFrame(columns=["hemi", "region", "F", "pval"])
    for hemi in df["hemi"].unique():
        for region in df["region"].unique():
            df_region = df[(df["hemi"] == hemi) & (df["region"] == region)]
            ancova = pg.ancova(
                data=df_region,
                dv=f"{measure}_change",
                between="dx_group",
                covar=["AGE_AT_VISIT", "SEX", "durationT2_T1_y"],
            )
            (F, pval) = ancova["F"].values[0], ancova["p-unc"].values[0]
            ancova_df.loc[len(ancova_df)] = [hemi, region, F, pval]

    ancova_df.rename(columns={"hemi": "hemisphere"}, inplace=True)
    filename = output_dir / f"ancova_longitudinal_{measure}.csv"
    ancova_df.to_csv(filename, index=False)

    return ancova_df


In [37]:
ancova_volume = compute_ancova("volume", df_cohort, force=True)
ancova_thickness = compute_ancova("thickness", df_cohort, force=True)
ancova_area = compute_ancova("area", df_cohort, force=True)

In [38]:
ancova_volume[ancova_volume["pval"] < 0.05].sort_values("F", ascending=False)

,hemisphere,region,F,pval
19,lh,pericalcarine,4.846042,0.028624


In [39]:
ancova_thickness[ancova_thickness["pval"] < 0.05].sort_values("F", ascending=False)

,hemisphere,region,F,pval


In [40]:
ancova_area[ancova_area["pval"] < 0.05].sort_values("F", ascending=False)

,hemisphere,region,F,pval
16,lh,parsopercularis,9.012454,0.002953
37,rh,bankssts,8.037979,0.004955
65,rh,superiortemporal,5.678216,0.017925
6,lh,inferiorparietal,4.294632,0.039258
50,rh,middletemporal,4.051039,0.045216
18,lh,parstriangularis,3.881635,0.049921


## Subcortical Volume

In [ ]:
def get_subcortical_volume_visit(cohort_df, visit):
    df = pd.read_csv("table_ieee/aseg.volume.tsv", sep="\t")
    df.rename(columns={"Measure:volume": "PATNO_id"}, inplace=True)

    # Validate visit parameter
    if visit not in [1, 2]:
        raise ValueError("Visit must be 1 or 2")

    df = df.melt(id_vars=["PATNO_id"], var_name="region", value_name="volume")

    visit_col = "first_visit" if visit == 1 else "second_visit"

    clinical_columns = [visit_col, "AGE_AT_VISIT", "SEX", "dx_group", "durationT2_T1_y"]

    merged_df = pd.merge(
        df,
        cohort_df[clinical_columns],
        left_on="PATNO_id",
        right_on=visit_col,
        how="inner",
    )

    # Clean up data types
    numeric_cols = ["volume", "AGE_AT_VISIT", "durationT2_T1_y"]
    for col in numeric_cols:
        if col in merged_df.columns:
            merged_df[col] = pd.to_numeric(merged_df[col], errors="coerce")

    return merged_df


def get_ancova_subcortical_volume_longitudinal(cohort_df):
    baseline_df = get_subcortical_volume_visit(cohort_df, visit=1)
    next_df = get_subcortical_volume_visit(cohort_df, visit=2)

    if baseline_df.empty:
        raise ValueError("No baseline data available")
    if next_df.empty:
        raise ValueError("No longitudinal data available")

    baseline_df["PATNO"] = baseline_df["PATNO_id"].str.split("_").str[0]
    next_df["PATNO"] = next_df["PATNO_id"].str.split("_").str[0]

    # Compute change
    columns_to_merge = ["PATNO", "region"]

    change_df = pd.merge(
        baseline_df,
        next_df,
        on=columns_to_merge,
        suffixes=("_baseline", "_next"),
    )

    if change_df.empty:
        raise ValueError("No matching records found between baseline and next visit")

    change_df["volume_change"] = (
        change_df["volume_next"] - change_df["volume_baseline"]
    ) / change_df["volume_baseline"]

    change_df.drop(columns=change_df.filter(regex="_baseline$").columns, inplace=True)
    change_df.rename(columns=lambda x: x.replace("_next", ""), inplace=True)

    ancova_df = pd.DataFrame(columns=["region", "F", "pval"])
    for region in tqdm(change_df["region"].unique()):
        df_region = change_df[change_df["region"] == region]
        try:
            ancova = pg.ancova(
                data=df_region,
                dv="volume_change",
                between="dx_group",
                covar=["AGE_AT_VISIT", "SEX", "durationT2_T1_y"],
            )
            (F, pval) = ancova["F"].values[0], ancova["p-unc"].values[0]
            ancova_df.loc[len(ancova_df)] = [region, F, pval]
        except Exception as e:
            print("Error:", e)
            print(f"Skipping region: {region}")
            ancova_df.loc[len(ancova_df)] = [region, np.nan, np.nan]

    filename = output_dir / "ancova_longitudinal_subcortical_volume.csv"
    ancova_df.to_csv(filename, index=False)

    return ancova_df


get_ancova_subcortical_volume_longitudinal(df_cohort)

  0%|                                                                                                                                                                                                 | 0/64 [00:00<?, ?it/s]

 45%|███████████████████████████████████████████████████████████████████████████████████▍                                                                                                    | 29/64 [00:00<00:00, 67.90it/s]

Error: r_matrix performs f_test for using dimensions that are asymptotically non-normal
Skipping region: Left-vessel


 62%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████                                                                     | 40/64 [00:00<00:00, 80.80it/s]

Error: r_matrix performs f_test for using dimensions that are asymptotically non-normal
Skipping region: Right-vessel
Error: r_matrix performs f_test for using dimensions that are asymptotically non-normal
Skipping region: 5th-Ventricle
Error: negative dimensions are not allowed
Skipping region: Left-WM-hypointensities
Error: negative dimensions are not allowed
Skipping region: Right-WM-hypointensities
Error: r_matrix performs f_test for using dimensions that are asymptotically non-normal
Skipping region: non-WM-hypointensities
Error: negative dimensions are not allowed
Skipping region: Left-non-WM-hypointensities
Error: negative dimensions are not allowed
Skipping region: Right-non-WM-hypointensities


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 71.86it/s]


,region,F,pval
0,Left-Lateral-Ventricle,0.010997,0.916567
1,Left-Inf-Lat-Vent,0.813998,0.367809
2,Left-Cerebellum-White-Matter,1.337337,0.248608
3,Left-Cerebellum-Cortex,1.681962,0.195860
4,Left-Thalamus,0.148749,0.700062
...,...,...,...
59,MaskVol-to-eTIV,0.582501,0.446053
60,lhSurfaceHoles,0.000234,0.987799
61,rhSurfaceHoles,0.002641,0.959058
62,SurfaceHoles,0.301228,0.583604
